In [3]:
# =========================
# 01 DATA PIPELINE (CORRECT GRID MATCH)
# =========================

import xarray as xr
import numpy as np
import rasterio
from shapely.geometry import shape, Point
import geopandas as gpd
from rasterio.features import shapes

# -------------------------
# LOAD ERA5
# -------------------------
ds = xr.open_dataset(r"P:\snowmelt_stochastic_model\data\raw\era5_chandra_temp_2018_2022.nc")

temp = ds['t2m'] - 273.15
temp = temp.rename({'valid_time': 'time'})

lats = ds.latitude.values
lons = ds.longitude.values

lon2d, lat2d = np.meshgrid(lons, lats)

# -------------------------
# LOAD BASIN MASK (RASTER)
# -------------------------
with rasterio.open(r"P:\snowmelt_stochastic_model\data\processed\basin_mask.tif") as src:
    basin = src.read(1)
    transform = src.transform
    crs = src.crs

# -------------------------
# CONVERT MASK → POLYGON
# -------------------------
mask = basin == 1

shapes_gen = shapes(basin.astype(np.uint8), mask=mask, transform=transform)
geoms = [shape(geom) for geom, value in shapes_gen if value == 1]

basin_gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs)
basin_gdf = basin_gdf.to_crs("EPSG:4326")  # convert to lat/lon

# -------------------------
# CREATE ERA5 MASK
# -------------------------
mask_era5 = np.zeros(lon2d.shape, dtype=bool)

for i in range(lat2d.shape[0]):
    for j in range(lat2d.shape[1]):
        point = Point(lon2d[i, j], lat2d[i, j])
        if basin_gdf.contains(point).any():
            mask_era5[i, j] = True

# -------------------------
# APPLY MASK
# -------------------------
temp_masked = temp.where(mask_era5)

temp_basin = temp_masked.mean(dim=['latitude', 'longitude'])

print(temp_basin)

<xarray.DataArray 't2m' (time: 1826)> Size: 7kB
array([-24.830536, -26.495514, -23.26825 , ..., -14.895351,  -9.919769,
       -24.231064], shape=(1826,), dtype=float32)
Coordinates:
    number   int64 8B ...
  * time     (time) datetime64[ns] 15kB 2018-01-01 2018-01-02 ... 2022-12-31
    expver   (time) <U4 29kB ...


In [4]:
temp_basin.to_netcdf(r"P:\snowmelt_stochastic_model\data\processed\temp_basin.nc")